# Confusion Matrix

## Imports

In [1]:
# IMPORTS
import os
import shutil
import time
import sys
import random

import numpy as np
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mpcol
from matplotlib.ticker import FormatStrFormatter

from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.axes_grid1 import make_axes_locatable

import pandas as pd
from dython.nominal import associations
from dython.nominal import identify_nominal_columns

from scipy import stats
from sklearn import datasets, mixture

from termcolor import colored, cprint
# Termcolor guide: https://pypi.org/project/termcolor/

from openpyxl import Workbook
from openpyxl import load_workbook

from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV

import seaborn as sns

# %matplotlib widget
%matplotlib inline
# %matplotlib notebook

# Try to use this website to use the explode feature, so we can see internal blocks and space everything out
# https://terbium.io/2017/12/matplotlib-3d/ 

# Use this website to make your GIFs - generally 50 delay per frame is good
# https://ezgif.com/maker



In [2]:
%pwd

'/Users/liamroy/Documents/Studies/Monash_31194990/PHD/Studies/Study_04/LLM_vocab_optimization/scripts'

## Create the Matrix

In [3]:
# Matrix Data Setup

excel_filepath = '/Users/liamroy/Documents/Studies/Monash_31194990/PHD/Studies/Study_04/LLM_vocab_optimization/data/proxy_validation/proxy_validation.xlsx'
excel_sheetname = 'PROXY_FINAL'

actual_class = ['WFI', 'AO', 'FO', 'NH', 'C']
predicted_class = ['WFI', 'AO', 'FO', 'NH', 'C', 'NON']

save_path = '/Users/liamroy/Documents/Studies/Monash_31194990/PHD/Studies/Study_04/LLM_vocab_optimization/plots/conf_matrix/'

plotter = None # None or True

In [4]:
def getConfusionMatrix(excel_file_path, excel_file_sheet, row_idx_start, row_idx_end, colum_idx, actual_classes, predicted_classes):
    # Number of classes is N
    # Creates the N x M grid for the predictions for each class
    # Rows are actual classes, columns are predicted classes
    df = pd.read_excel(excel_file_path, sheet_name = excel_file_sheet)

    data = df.iloc[row_idx_start-2:row_idx_end-1, colum_idx-1].values  # This gives you a 1D array of 30 elements
    
    # Round the percentages down to nearest iteger and convert to integers
    data_numeric = np.nan_to_num(data.astype(float))
    confMat_percent = np.round(data_numeric * 100).astype(int)
    # Reshape the data into desired shape
    # Example: if you want 5 rows and 6 columns
    confMatrix = confMat_percent.reshape(len(actual_classes), len(predicted_classes))

    # Print the reshaped data
    # print(confMatrix)

    # Create a confusion matrix from the reshaped data
    return confMatrix

In [5]:
GPT4_true_confMat = getConfusionMatrix(excel_file_path=excel_filepath, 
                                       excel_file_sheet=excel_sheetname, 
                                       row_idx_start=2, 
                                       row_idx_end=31, 
                                       colum_idx=1, 
                                       actual_classes=actual_class, 
                                       predicted_classes=predicted_class)
print(f"GPT4_true_confMat:\n{GPT4_true_confMat}\n\n")

GPT4_proxy_confMat = getConfusionMatrix(excel_file_path=excel_filepath, 
                                       excel_file_sheet=excel_sheetname, 
                                       row_idx_start=2, 
                                       row_idx_end=31, 
                                       colum_idx=2, 
                                       actual_classes=actual_class, 
                                       predicted_classes=predicted_class)
print(f"GPT4_proxy_confMat:\n{GPT4_proxy_confMat}\n\n")

human_true_confMat = getConfusionMatrix(excel_file_path=excel_filepath, 
                                       excel_file_sheet=excel_sheetname, 
                                       row_idx_start=32, 
                                       row_idx_end=61, 
                                       colum_idx=1, 
                                       actual_classes=actual_class, 
                                       predicted_classes=predicted_class)
print(f"human_true_confMat:\n{human_true_confMat}\n\n")

human_proxy_confMat = getConfusionMatrix(excel_file_path=excel_filepath, 
                                       excel_file_sheet=excel_sheetname, 
                                       row_idx_start=32, 
                                       row_idx_end=61, 
                                       colum_idx=2, 
                                       actual_classes=actual_class, 
                                       predicted_classes=predicted_class)
print(f"human_proxy_confMat:\n{human_proxy_confMat}\n\n")

random_true_confMat = getConfusionMatrix(excel_file_path=excel_filepath, 
                                       excel_file_sheet=excel_sheetname, 
                                       row_idx_start=62, 
                                       row_idx_end=91, 
                                       colum_idx=1, 
                                       actual_classes=actual_class, 
                                       predicted_classes=predicted_class)
print(f"GPT4_true_confMat:\n{random_true_confMat}\n\n")

random_proxy_confMat = getConfusionMatrix(excel_file_path=excel_filepath, 
                                       excel_file_sheet=excel_sheetname, 
                                       row_idx_start=62, 
                                       row_idx_end=91, 
                                       colum_idx=2, 
                                       actual_classes=actual_class, 
                                       predicted_classes=predicted_class)
print(f"GPT4_proxy_confMat:\n{random_proxy_confMat}\n\n")


GPT4_true_confMat:
[[73  2  2  5  7 12]
 [ 8 33 50  3  3  2]
 [ 2 57 40  0  0  2]
 [15 10  3 52 12  8]
 [ 5  8  0 52 32  3]]


GPT4_proxy_confMat:
[[80 10  0  0  5  5]
 [ 0 65 20  0 10  5]
 [ 0 90 10  0  0  0]
 [ 0  0  0 30 70  0]
 [ 5  0  0  0 90  5]]


human_true_confMat:
[[65  8  8  5  7  7]
 [ 0 55 43  0  2  0]
 [ 0 50 48  2  0  0]
 [57  3  5 18 15  2]
 [60  2  3  7 27  2]]


human_proxy_confMat:
[[40 15  0 25 20  0]
 [ 0 80 10  5  5  0]
 [ 0 70 20  5  5  0]
 [ 0  0  0 30 70  0]
 [10 10  0 40 40  0]]


GPT4_true_confMat:
[[ 2 15 25 15 38  5]
 [13  2  3 58 12 12]
 [ 0 30 17 28 23  2]
 [ 0 45 15  7 23 10]
 [15  0  5 57 18  5]]


GPT4_proxy_confMat:
[[ 0 20 20 10 50  0]
 [ 0  5  0 40 55  0]
 [ 0 30 20  0 50  0]
 [ 0 15 35  0 50  0]
 [ 5  0  0 20 75  0]]




In [16]:

def plot_matrix(matrix_list, condition_name_list, save_filename_list, save_path, type_of_matrix, real_or_proxy_list):

    """
    Plots confusion matrices from a list of matrices and saves them to specified paths.
    
    Args:
        matrix_list (list): List of confusion matrices to plot.
        condition_name_list (list): List of condition names corresponding to each matrix.
        save_filename_list (list): List of filenames to save the plots.
        save_path (str): Path where the plots will be saved.
        real_or_proxy_list (list): List indicating whether the state estimation is real or proxy.
        type_of_matrix (str): Type of matrix to plot, either 'conf_matrix' or 'delta_matrix'.
    """
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    for x in range(len(matrix_list)):

        save_str = save_path + save_filename_list[x]

        if type_of_matrix == 'conf_matrix':
            # Title and Axes Labels
            plot_title = f'\n{real_or_proxy_list[x]} State Estimation Confusion\nMatrix for {condition_name_list[x]} Generated Poses\n'

        elif type_of_matrix == 'delta_matrix':
            # Title and Axes Labels
            plot_title = f'\nDelta State Estimation Between\nProxy and Real for {condition_name_list[x]} Generated Poses\n'

        else:
            raise ValueError(f"Invalid type_of_matrix: {type_of_matrix}")


        # Plotting Setup
        fig, axs = plt.subplots(figsize=(12, 12))

        # Set the normalization range from 0 to 40 and apply it to the imshow function
        norm = mpcol.Normalize(vmin=0, vmax=100)
        im = axs.imshow(matrix_list[x], cmap='viridis', norm=norm)

        # Title and Axes Labels
        axs.set_title(plot_title, size=32)
        axs.set_xlabel(f"Robot State Selected by {real_or_proxy_list[x]}", size=30)
        axs.set_ylabel("True Robot State\n", size=32)

        # Show all ticks and label them with the respective list entries
        axs.set_yticks(np.arange(len(actual_class)), labels=actual_class)
        axs.set_xticks(np.arange(len(predicted_class)), labels=predicted_class)

        # Rotate the tick labels and set their alignment.
        plt.setp(axs.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor", size=28)
        plt.setp(axs.get_yticklabels(), rotation=45, ha="right", rotation_mode="anchor", size=28)

        # Make the Sidebar
        divider = make_axes_locatable(axs)
        cax = divider.append_axes("right", size="5%", pad=0.05)
        cbar = plt.colorbar(im, cax=cax)
        cbar.ax.tick_params(labelsize=26)


        # Loop over data dimensions and create text annotations.
        for i in range(len(actual_class)):
            for j in range(len(predicted_class)):
                text = axs.text(j, i, f"{matrix_list[x][i, j]:.0f}%",
                                ha="center", va="center", color="w", size=28)

        fig.tight_layout()

        # Save the fig
        plt.savefig(save_str, bbox_inches='tight', pad_inches=0.25)

        # Show the fig if plotter is not set to None
        if plotter:
            plt.show()

        if plotter == None:
            print("Plots closed.")
            plt.close()



## Plotting Confusion Matrices

In [17]:
conf_matrix_list = [GPT4_true_confMat, 
                    GPT4_proxy_confMat, 
                    human_true_confMat, 
                    human_proxy_confMat, 
                    random_true_confMat, 
                    random_proxy_confMat]

conf_save_filename_list = [
    'GPT4o_real.png', 
    'GPT4o_proxy.png', 
    'Human_real.png', 
    'Human_proxy.png', 
    'Random_real.png', 
    'Random_proxy.png'
]

conf_condition_name_list = [
    'GPT4o', 
    'GPT4o', 
    'Human', 
    'Human', 
    'Random', 
    'Random'
]

conf_real_or_proxy_list = [
    'User', 
    'Proxy', 
    'User', 
    'Proxy', 
    'User', 
    'Proxy'
]

plot_matrix(matrix_list=conf_matrix_list, 
            condition_name_list=conf_condition_name_list, 
            save_filename_list=conf_save_filename_list, 
            save_path=save_path, 
            type_of_matrix='conf_matrix', 
            real_or_proxy_list=conf_real_or_proxy_list)



Plots closed.
Plots closed.
Plots closed.
Plots closed.
Plots closed.
Plots closed.


## Now Create Plots to Show Difference

In [18]:
# Create a plot to show the difference between the proxy and real matrixies for the three conditions (GPT4o, human, random)

delta_matrix_GPT4o = abs(GPT4_proxy_confMat - GPT4_true_confMat)
delta_matrix_human = abs(human_proxy_confMat - human_true_confMat)
delta_matrix_random = abs(random_proxy_confMat - random_true_confMat)

print(delta_matrix_GPT4o, "\n")
print(delta_matrix_human, "\n")
print(delta_matrix_random, "\n")

delta_matrix_list = [delta_matrix_GPT4o, 
                     delta_matrix_human, 
                     delta_matrix_random]
   
delta_condition_name_list = [
    'GPT4o', 
    'Human', 
    'Random', 
]

delta_save_filename_list = [
    'GPT4o_delta.png', 
    'Human_delta.png', 
    'Random_delta.png', 
]

conf_real_or_proxy_list = [
    'User/Proxy',
    'User/Proxy', 
    'User/Proxy', 
]


plot_matrix(matrix_list=delta_matrix_list, 
            condition_name_list=delta_condition_name_list, 
            save_filename_list=delta_save_filename_list, 
            save_path=save_path, 
            type_of_matrix='delta_matrix',
            real_or_proxy_list=conf_real_or_proxy_list)


[[ 7  8  2  5  2  7]
 [ 8 32 30  3  7  3]
 [ 2 33 30  0  0  2]
 [15 10  3 22 58  8]
 [ 0  8  0 52 58  2]] 

[[25  7  8 20 13  7]
 [ 0 25 33  5  3  0]
 [ 0 20 28  3  5  0]
 [57  3  5 12 55  2]
 [50  8  3 33 13  2]] 

[[ 2  5  5  5 12  5]
 [13  3  3 18 43 12]
 [ 0  0  3 28 27  2]
 [ 0 30 20  7 27 10]
 [10  0  5 37 57  5]] 

Plots closed.
Plots closed.
Plots closed.
